In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
from scipy.signal import find_peaks, savgol_filter
from scipy.interpolate import interp1d, UnivariateSpline
from tqdm.notebook import tqdm
import matplotlib.patches as patches
from ipywidgets import interact, IntRangeSlider, IntSlider
import random

In [ ]:
# 1. Configuración de carpetas
input_folder = 'IMAGES'
output_folder = 'RESULTADOS_ECG'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider

def analisis_recorte(n_imagen, p_sup, p_inf, p_izq, p_der):
    archivo_muestra = f"image ({n_imagen}).png"
    path = os.path.join(input_folder, archivo_muestra)
    
    img = cv2.imread(path)
    if img is None:
        print(f"❌ No se encontró: {archivo_muestra}")
        return

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    alto, ancho = img.shape[:2]

    # Cálculo de píxeles
    l_sup, l_inf = int(alto * p_sup), int(alto * p_inf)
    l_izq, l_der = int(ancho * p_izq), int(ancho * p_der)

    # Figura más ancha (15 pulgadas de ancho)
    plt.figure(figsize=(15, 7))
    plt.imshow(img_rgb)
    ax = plt.gca()

    # Líneas de corte dinámicas
    ax.axhline(l_sup, color="red", linestyle="--", linewidth=2)
    ax.axhline(l_inf, color="red", linestyle="--", linewidth=2)
    ax.axvline(l_izq, color="red", linestyle="--", linewidth=2)
    ax.axvline(l_der, color="red", linestyle="--", linewidth=2)

    # Sombreado del área que se ELIMINA (oscurecemos lo que no se usa)
    # Esto ayuda a centrar la vista en el ECG
    rect_top = patches.Rectangle((0,0), ancho, l_sup, color='black', alpha=0.5)
    rect_bot = patches.Rectangle((0,l_inf), ancho, alto-l_inf, color='black', alpha=0.5)
    ax.add_patch(rect_top)
    ax.add_patch(rect_bot)

    plt.title(f"Vista: {archivo_muestra} | Área útil: {l_der-l_izq}x{l_inf-l_sup}px", fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Interfaz interactiva completa
interact(
    analisis_recorte, 
    n_imagen=IntSlider(min=1, max=270, step=1, value=1, description="Imagen:"),
    p_sup=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.42, description="Corte Sup:"),
    p_inf=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.50, description="Corte Inf:"),
    p_izq=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.24, description="Corte Izq:"),
    p_der=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.50, description="Corte Der:")
)

In [ ]:
# Escritura a mano de los parámetros óptimos encontrados para cada imagen, poniendo todos por defecto a los mismos valores
config_recortes = {i: (0.42, 0.50, 0.24, 0.50) for i in range(1, 271)}
n_sup = 0.42
n_inf = 0.50
n_izq = 0.24
n_der = 0.50
# Ejemplo de cómo modificar un valor específico (si se quisiera ajustar alguno manualmente)
config_recortes[1] = (n_sup, n_inf, n_izq, 0.44)
config_recortes[2] = (n_sup, n_inf, 0.49, 0.69)
config_recortes[3] = (n_sup, n_inf, 0.26, 0.47)
config_recortes[4] = (n_sup, n_inf, 0.37, 0.70)
config_recortes[5] = (n_sup, n_inf, 0.27, 0.48) 
config_recortes[6] = (n_sup, n_inf, 0.16, 0.51)
config_recortes[7] = (n_sup, n_inf, 0.30, 0.56)
config_recortes[9] = (n_sup, n_inf, 0.35, 0.63) 
config_recortes[10] = (n_sup, n_inf, 0.29, 0.59)
config_recortes[12] = (n_sup, n_inf, 0.34, 0.62) 
config_recortes[13] = (n_sup, n_inf, 0.25, 0.51)
config_recortes[14] = (n_sup, n_inf, 0.30, 0.54)
config_recortes[15] = (n_sup, n_inf, 0.32, 0.61)
config_recortes[16] = (n_sup, n_inf, 0.32, 0.61)
config_recortes[17] = (0.41, n_inf, 0.39, 0.68)
config_recortes[19] = (n_sup, n_inf, 0.19, n_der)
config_recortes[20] = (n_sup, n_inf, 0.34, 0.66)
config_recortes[21] = (0.40, n_inf, n_izq, n_der)
config_recortes[22] = (n_sup, n_inf, 0.30, 0.61)
config_recortes[23] = (n_sup, n_inf, 0.21, 0.43)
config_recortes[25] = (n_sup, n_inf, 0.21, 0.51)
config_recortes[26] = (n_sup, n_inf, n_izq, 0.60)
config_recortes[27] = (n_sup, n_inf, 0.37, 0.63)
config_recortes[28] = (n_sup, n_inf, 0.18, 0.45)
config_recortes[29] = (n_sup, n_inf, 0.34, 0.58)
config_recortes[30] = (n_sup, n_inf, n_izq, 0.52)
config_recortes[31] = (n_sup, n_inf, 0.30, 0.59)
config_recortes[32] = (n_sup, n_inf, 0.18, n_der)
config_recortes[33] = (n_sup, n_inf, 0.35, 0.62)
config_recortes[34] = (n_sup, n_inf, 0.22, 0.47)
config_recortes[36] = (n_sup, n_inf, 0.21, n_der)
config_recortes[38] = (n_sup, n_inf, n_izq, 0.53)
config_recortes[39] = (n_sup, n_inf, 0.31, 0.57)
config_recortes[40] = (n_sup, n_inf, 0.22, n_der)
config_recortes[41] = (n_sup, n_inf, n_izq, 0.53)
config_recortes[42] = (0.41, n_inf, n_izq, n_der)
config_recortes[43] = (n_sup, n_inf, 0.35, 0.66)
config_recortes[45] = (n_sup, n_inf, 0.21, n_der)
config_recortes[46] = (n_sup, n_inf, n_izq, 0.68)
config_recortes[48] = (n_sup, n_inf, 0.19, 0.56)
config_recortes[49] = (n_sup, n_inf, 0.19, 0.58)
config_recortes[50] = (n_sup, n_inf, 0.32, 0.53)
config_recortes[51] = (n_sup, n_inf, 0.32, 0.56)
config_recortes[52] = (n_sup, n_inf, 0.32, 0.56)
config_recortes[53] = (n_sup, n_inf, 0.30, 0.55)
config_recortes[54] = (n_sup, n_inf, n_izq, 0.57)
config_recortes[55] = (n_sup, n_inf, 0.20, 0.46)
config_recortes[56] = (n_sup, n_inf, 0.19, 0.45)
config_recortes[57] = (n_sup, n_inf, 0.22, 0.46)
config_recortes[59] = (n_sup, n_inf, 0.32, 0.58)
config_recortes[60] = (n_sup, n_inf, 0.38, 0.71)
config_recortes[61] = (n_sup, n_inf, 0.21, n_der)
config_recortes[62] = (n_sup, n_inf, 0.19, n_der)
config_recortes[63] = (n_sup, n_inf, n_izq, 0.57)
config_recortes[65] = (n_sup, n_inf, 0.22, n_der)
config_recortes[67] = (n_sup, n_inf, 0.22, n_der)
config_recortes[68] = (n_sup, n_inf, 0.34, 0.68)
config_recortes[70] = (n_sup, n_inf, 0.20, 0.44)
config_recortes[71] = (n_sup, n_inf, 0.34, 0.70)
config_recortes[74] = (n_sup, n_inf, 0.33, 0.58)
config_recortes[75] = (n_sup, n_inf, 0.36, 0.68)
config_recortes[76] = (n_sup, n_inf, 0.32, 0.56)
config_recortes[79] = (n_sup, n_inf, 0.18, 0.42)
config_recortes[80] = (n_sup, n_inf, 0.40, 0.70)
config_recortes[82] = (n_sup, n_inf, 0.18, 0.43)
config_recortes[83] = (n_sup, n_inf, 0.45, 0.76)
config_recortes[84] = (n_sup, n_inf, 0.40, 0.62)
config_recortes[85] = (n_sup, n_inf, 0.33, 0.55)
config_recortes[86] = (n_sup, n_inf, 0.30, 0.60)
config_recortes[87] = (n_sup, n_inf, 0.55, 0.81)
config_recortes[88] = (n_sup, n_inf, 0.39, 0.70)
config_recortes[89] = (n_sup, n_inf, 0.22, 0.48)
config_recortes[91] = (n_sup, n_inf, n_izq, 0.50)
config_recortes[92] = (n_sup, n_inf, 0.35, 0.61)
config_recortes[93] = (n_sup, n_inf, 0.40, 0.67)
config_recortes[94] = (n_sup, n_inf, 0.36, 0.58)
config_recortes[95] = (n_sup, n_inf, 0.34, 0.60)
config_recortes[96] = (n_sup, n_inf, n_izq, 0.56)
config_recortes[98] = (n_sup, n_inf, 0.41, 0.73)
config_recortes[99] = (n_sup, n_inf, 0.32, 0.57)
config_recortes[100] = (0.41, n_inf, 0.36, 0.62)
config_recortes[101] = (n_sup, n_inf, 0.25, 0.52)
config_recortes[103] = (n_sup, n_inf, 0.28, 0.54)
config_recortes[104] = (n_sup, n_inf, 0.20, 0.50)
config_recortes[106] = (0.41, n_inf, 0.28, 0.52)
config_recortes[108] = (n_sup, n_inf, 0.33, 0.58)
config_recortes[109] = (n_sup, n_inf, n_izq, 0.67)
config_recortes[110] = (n_sup, n_inf, 0.19, 0.52)
config_recortes[111] = (n_sup, n_inf, 0.48, 0.75)
config_recortes[112] = (n_sup, n_inf, 0.39, 0.59)
config_recortes[113] = (n_sup, n_inf, 0.30, 0.53)
config_recortes[115] = (n_sup, n_inf, 0.30, 0.61)
config_recortes[117] = (n_sup, n_inf, 0.17, n_der)
config_recortes[118] = (n_sup, n_inf, 0.19, n_der)
config_recortes[122] = (n_sup, n_inf, 0.17, 0.45)
config_recortes[123] = (n_sup, n_inf, 0.32, 0.63)
config_recortes[124] = (n_sup, n_inf, n_izq, 0.55)
config_recortes[125] = (n_sup, n_inf, 0.38, 0.62)
config_recortes[126] = (n_sup, n_inf, 0.35, 0.62)
config_recortes[127] = (n_sup, n_inf, 0.29, 0.53)
config_recortes[128] = (n_sup, n_inf, 0.33, 0.61)
config_recortes[129] = (n_sup, n_inf, 0.28, 0.59)
config_recortes[130] = (n_sup, n_inf, 0.28, 0.52)
config_recortes[131] = (n_sup, n_inf, 0.16, 0.46)
config_recortes[132] = (n_sup, n_inf, 0.16, 0.46)
config_recortes[134] = (n_sup, n_inf, 0.37, 0.67)
config_recortes[136] = (n_sup, n_inf, 0.32, 0.58)
config_recortes[136] = (n_sup, n_inf, 0.34, 0.58)
config_recortes[138] = (n_sup, n_inf, 0.30, 0.57)
config_recortes[139] = (n_sup, n_inf, 0.26, 0.53)
config_recortes[140] = (n_sup, n_inf, n_izq, 0.52)
config_recortes[141] = (n_sup, n_inf, 0.21, n_der)
config_recortes[142] = (n_sup, n_inf, 0.28, 0.54)
config_recortes[143] = (n_sup, n_inf, n_izq, n_der)
config_recortes[144] = (n_sup, n_inf, n_izq, n_der)
config_recortes[145] = (n_sup, n_inf, n_der, 0.72)
config_recortes[146] = (n_sup, n_inf, 0.38, 0.64)
config_recortes[147] = (n_sup, n_inf, 0.22, 0.47)
config_recortes[148] = (n_sup, n_inf, 0.39, 0.65)
config_recortes[149] = (n_sup, n_inf, 0.39, 0.72)
config_recortes[150] = (n_sup, n_inf, 0.39, 0.58)
config_recortes[151] = (n_sup, n_inf, n_izq, 0.60)
config_recortes[152] = (n_sup, n_inf, n_izq, 0.60)
config_recortes[153] = (n_sup, n_inf, n_izq, 0.61)
config_recortes[154] = (n_sup, n_inf, 0.21, 0.46)
config_recortes[156] = (n_sup, n_inf, 0.33, 0.69)
config_recortes[157] = (n_sup, n_inf, 0.17, 0.49)
config_recortes[159] = (n_sup, n_inf, 0.22, 0.57)
config_recortes[161] = (n_sup, n_inf, 0.22, 0.54)
config_recortes[162] = (n_sup, n_inf, 0.22, 0.57)
config_recortes[163] = (n_sup, n_inf, 0.37, 0.67)
config_recortes[165] = (n_sup, n_inf, 0.37, 0.70)
config_recortes[167] = (n_sup, n_inf, 0.17, n_der)
config_recortes[169] = (n_sup, n_inf, 0.34, 0.57)
config_recortes[170] = (n_sup, n_inf, 0.22, 0.45)
config_recortes[171] = (n_sup, n_inf, 0.25, 0.54)
config_recortes[172] = (n_sup, n_inf, 0.35, 0.61)
config_recortes[173] = (n_sup, n_inf, 0.21, 0.49)
config_recortes[174] = (n_sup, n_inf, n_izq, 0.54)
config_recortes[175] = (n_sup, n_inf, 0.38, 0.58)
config_recortes[176] = (n_sup, n_inf, n_izq, 0.54)
config_recortes[177] = (n_sup, n_inf, 0.33, 0.70)
config_recortes[178] = (n_sup, n_inf, n_izq, 0.54)
config_recortes[179] = (n_sup, n_inf, n_izq, 0.54)
config_recortes[181] = (n_sup, n_inf, 0.16, n_der)
config_recortes[182] = (n_sup, n_inf, n_izq, 0.56)
config_recortes[183] = (n_sup, n_inf, 0.22, 0.56)
config_recortes[184] = (n_sup, n_inf, 0.16, 0.51)
config_recortes[185] = (n_sup, n_inf, 0.16, 0.51)
config_recortes[187] = (n_sup, n_inf, 0.16, n_der)
config_recortes[188] = (n_sup, n_inf, 0.16, n_der)
config_recortes[189] = (n_sup, n_inf, n_izq, 0.53)
config_recortes[190] = (n_sup, n_inf, 0.39, 0.72)
config_recortes[191] = (n_sup, n_inf, n_izq, 0.54)
config_recortes[192] = (n_sup, n_inf, n_izq, 0.54)
config_recortes[193] = (n_sup, n_inf, 0.16, 0.38)
config_recortes[194] = (n_sup, n_inf, 0.36, 0.59)
config_recortes[195] = (n_sup, n_inf, 0.39, 0.68)
config_recortes[196] = (n_sup, n_inf, 0.36, 0.68)
config_recortes[197] = (n_sup, n_inf, 0.36, 0.68)
config_recortes[198] = (n_sup, n_inf, 0.30, 0.53)
config_recortes[199] = (n_sup, n_inf, 0.30, 0.59)
config_recortes[200] = (n_sup, n_inf, 0.33, 0.65)
config_recortes[201] = (n_sup, n_inf, 0.33, 0.53)
config_recortes[202] = (n_sup, n_inf, n_izq, 0.52)
config_recortes[203] = (n_sup, n_inf, 0.40, 0.68)
config_recortes[204] = (n_sup, n_inf, n_izq, 0.59)
config_recortes[205] = (n_sup, n_inf, 0.39, 0.78)
config_recortes[206] = (n_sup, n_inf, 0.57, 0.79)
config_recortes[207] = (n_sup, n_inf, 0.22, n_der)
config_recortes[208] = (n_sup, n_inf, 0.39, 0.70)
config_recortes[209] = (n_sup, n_inf, 0.39, 0.71)
config_recortes[210] = (n_sup, n_inf, 0.36, 0.70)
config_recortes[211] = (n_sup, n_inf, n_izq, 0.60)
config_recortes[212] = (n_sup, n_inf, 0.20, 0.51)
config_recortes[215] = (n_sup, n_inf, 0.18, 0.44)
config_recortes[216] = (n_sup, n_inf, n_izq, 0.54)
config_recortes[217] = (n_sup, n_inf, 0.19, 0.47)
config_recortes[219] = (n_sup, n_inf, n_izq, 0.57)
config_recortes[220] = (n_sup, n_inf, n_izq, 0.61)
config_recortes[221] = (n_sup, n_inf, n_izq, 0.61)
config_recortes[222] = (n_sup, n_inf, 0.35, 0.67)
config_recortes[223] = (n_sup, n_inf, 0.35, 0.63)
config_recortes[251] = (n_sup, n_inf, n_izq, 0.53)
config_recortes[253] = (n_sup, n_inf, 0.28, 0.57)
config_recortes[257] = (n_sup, n_inf, 0.27, n_der)
config_recortes[260] = (n_sup, n_inf, 0.30, 0.62)
config_recortes[262] = (n_sup, n_inf, 0.30, 0.62)
config_recortes[263] = (n_sup, n_inf, n_izq, 0.58)
config_recortes[264] = (n_sup, n_inf, n_izq, 0.62)
config_recortes[265] = (n_sup, n_inf, 0.36, 0.61)
config_recortes[266] = (n_sup, n_inf, 0.36, 0.64)
config_recortes[267] = (n_sup, n_inf, 0.30, 0.59)
config_recortes[268] = (n_sup, n_inf, 0.21, 0.46)
config_recortes[269] = (n_sup, n_inf, 0.25, 0.51)
config_recortes[270] = (n_sup, n_inf, 0.27, 0.55)


In [ ]:
# 1. Cargamos una imagen de prueba (puedes cambiar el índice para probar con otras)
n_test = 5
test_path = os.path.join(input_folder, f'image ({n_test}).png')
img_test = cv2.imread(test_path)

if img_test is not None:
    alto, ancho = img_test.shape[:2]
    
    # 2. Aplicamos el recorte EXACTO que definiste en config_recortes para esa imagen
    # Usamos los valores por defecto que tienes en tu celda 4
    p_sup, p_inf, p_izq, p_der = config_recortes[n_test]
    crop = img_test[int(alto*p_sup):int(alto*p_inf), int(ancho*p_izq):int(ancho*p_der)]
    
    # Pre-procesado: conversión a HSV
    hsv_test = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

    def update_mask(h_range, s_range, v_range):
        bajo = np.array([h_range[0], s_range[0], v_range[0]])
        alto_val = np.array([h_range[1], s_range[1], v_range[1]])
        
        # --- MISMO PROCESADO QUE EN LA EJECUCIÓN ---
        # Aplicamos el desenfoque para reducir ruido antes de la máscara
        blur = cv2.GaussianBlur(hsv_test, (5, 5), 0)
        mask = cv2.inRange(blur, bajo, alto_val)
        # -------------------------------------------
        
        plt.figure(figsize=(15, 5))
        
        # Mostrar Recorte Original
        plt.subplot(1, 3, 1)
        plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        plt.title(f"Recorte Original (Imagen {n_test})")
        plt.axis('off')
        
        # Mostrar Máscara Resultante
        plt.subplot(1, 3, 2)
        plt.imshow(mask, cmap='gray')
        plt.axvline(17, color='red', linestyle='--', label='Centro de la Señal')
        plt.axvline(219, color='red', linestyle='--', label='Centro de la Señal')
        plt.title(f"Máscara (Lo que el código 've')\nBajo: {bajo} | Alto: {alto_val}")
        plt.axis('off')

        # Quedarnos con el valor medio de cada columna para simular la extracción de la señal ECG (simplificado)
        plt.subplot(1, 3, 3)
        signal = []
        for col in range(mask.shape[1]):
            indices = np.where(mask[:, col] > 0)[0]
            if len(indices) > 0:
                signal.append(np.median(indices))
        signal = np.array(signal)
        plt.plot(-signal+np.mean(signal), color='blue')  # Centrar la señal en torno a su media para mejor visualización
        plt.title("Señal ECG Simulada")
        plt.xlabel("Columna")
        plt.ylabel("Valor Medio")

        # Añadir una linea horizontal que sea el valor medio de la señal para visualizar mejor el ECG
        if len(signal) > 0:
            plt.axhline(np.max(-signal+np.mean(signal)), color='green', linestyle='--', label='Valor Máximo')
            plt.axhline(np.min(-signal+np.mean(signal)), color='orange', linestyle='--', label='Valor Mínimo')
            plt.axvline(17, color='red', linestyle='--', label='Centro de la Señal')
            plt.axvline(219, color='red', linestyle='--', label='Centro de la Señal')
            plt.legend()
        plt.show()

    # Interfaz con los valores actuales que tienes en tu código (celda 6)
    interact(update_mask, 
             h_range=IntRangeSlider(value=[38, 89], min=0, max=180, step=1, description='Hue'),
             s_range=IntRangeSlider(value=[130, 255], min=0, max=255, step=1, description='Sat'),
             v_range=IntRangeSlider(value=[74, 255], min=0, max=255, step=1, description='Val'))
else:
    print(f"❌ No se pudo cargar la imagen {test_path} para la prueba.")

In [ ]:
# Parámetros de color (puedes ajustarlos según tus pruebas previas)
verde_bajo = np.array([38, 100, 70])
verde_alto = np.array([89, 255, 255])

In [ ]:
for f in os.listdir(output_folder):
    os.remove(os.path.join(output_folder, f))

cnt = 0
limite = None
datos_todas_imagenes = []
señales_malas = []
filenames = sorted(os.listdir(input_folder), key=lambda x: int(x.split('(')[1].split(')')[0]))

for filename in tqdm(filenames, desc="Procesando imágenes"):
    if filename.lower().endswith(('.png')) and (limite is None or cnt < limite):

        img_path = os.path.join(input_folder, filename)
        img = cv2.imread(img_path)
        num_img = int(filename.split('(')[1].split(')')[0])
        p_sup, p_inf, p_izq, p_der = config_recortes.get(num_img, (0.42, 0.50, 0.24, 0.50))  # Valores por defecto si no se encuentra

        alto, ancho = img.shape[:2]
        crop = img[int(alto*p_sup):int(alto*p_inf), int(ancho*p_izq):int(ancho*p_der)]
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        blur = cv2.GaussianBlur(hsv, (5, 5), 0)
        mask = cv2.inRange(blur, verde_bajo, verde_alto)
        mask = cv2.medianBlur(mask, 3)
        
        signal = []
        for col in range(mask.shape[1]):
            indices = np.where(mask[:, col] > 0)[0]

            if len(indices) > 0:
                signal.append(np.median(indices))
            else:
                signal.append(signal[-1] if signal else 0)
        
        signal = np.array(signal)
        signal = np.max(signal) - signal
        largo = len(signal)
        rango = np.max(signal) - np.min(signal)

        try:
            signal_centrada = signal - np.mean(signal)
            if np.abs(np.max(signal_centrada)) > np.abs(np.min(signal_centrada)):
                rango = np.max(signal_centrada) - np.min(signal_centrada)
                prom_dinamica = rango * 0.1
                picos, properties = find_peaks(signal_centrada, prominence=prom_dinamica, distance=largo*0.75)

            else:
                rango = np.max(signal_centrada) - np.min(signal_centrada)
                prom_dinamica = rango * 0.1
                picos, properties = find_peaks(-signal_centrada, prominence=prom_dinamica, distance=largo*0.75)

        except Exception as e:
            print(f"Error encontrando picos en {filename}: {e}")
            picos = []

        if len(picos) >= 2:
            pico_p = picos[0]
            pico_u = picos[-1]
            picos_finales = [pico_p, pico_u]
            signal_recortada = signal[picos_finales[0]:picos_finales[1]]
            cnt += 1

            x_old = np.linspace(0, 1, len(signal_recortada))
            x_new = np.linspace(0, 1, 1001)

            f_interp = UnivariateSpline(x_old, signal_recortada) 
            signal_1001 = f_interp(x_new)

            datos_imagen = {
                        'filename': filename,
                        'signal': signal_1001
                    }
            datos_todas_imagenes.append(datos_imagen)
            
            plt.figure(figsize=(10, 4))
            plt.plot(signal_1001, color='blue')
            plt.title(f"Señal ECG - {filename}")
            output_path = os.path.join(output_folder, f"{filename.split('.')[0]}_ecg.png")
            plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
            plt.close()

        else:
            print(f"⚠️ No se encontraron suficientes picos en {filename}. Se encontraron {len(picos)} picos en las posiciones {picos}.")

            x_old = np.linspace(0, 1, len(signal))
            x_new = np.linspace(0, 1, 1001)

            f_interp = UnivariateSpline(x_old, signal)
            signal_1001_mala = f_interp(x_new)

            datos_mala = {
                'filename': filename,
                'signal': signal_1001_mala
                }
            señales_malas.append(datos_mala)
            continue
            
  

df = pd.DataFrame(datos_todas_imagenes)
csv_path = os.path.join(output_folder, "señales_ecg.csv")
df.to_csv(csv_path, index=False)

malas = pd.DataFrame(señales_malas)
malas_csv_path = os.path.join(output_folder, "señales_malas.csv")
malas.to_csv(malas_csv_path, index=False)

print(f"✅ Procesadas {cnt} imágenes. Señales guardadas en '{output_folder}'.")
    

In [ ]:
# Comprobacion de las imágenes resultantes
n = random.randint(0, len(datos_todas_imagenes)-1)

for datos in datos_todas_imagenes[0:10]: 
    filename = datos['filename']
    signal = datos['signal']
    
    # Cargar la imagen original
    img = cv2.imread(os.path.join(input_folder, filename))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Recortar la imagen
    alto, ancho = img.shape[:2]
    crop = img[int(alto*0.3):int(alto*0.7), :]
    
    # Mostrar la imagen original y la señal extraída
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img_rgb)
    plt.title(f"Imagen Original - {filename}")
    
    plt.subplot(1, 2, 2)
    plt.plot(signal, color='green')
    plt.title(f"Señal ECG extraída - {filename}")
    plt.xlabel("Posición horizontal (px)")
    plt.ylabel("Posición vertical media (px)")
    plt.grid()
    
    plt.tight_layout()
    plt.show()
    

In [ ]:
# Analizamos las señales malas y recortamos manualmente las que se puedan salvar
for datos in señales_malas[:5]:
    filename = datos['filename']
    signal = datos['signal']
    
    # Cargar la imagen original
    img = cv2.imread(os.path.join(input_folder, filename))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Mostrar la imagen original y la señal extraída
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img_rgb)
    plt.title(f"Imagen Original - {filename}")
    
    plt.subplot(1, 2, 2)
    plt.plot(signal, color='red')
    plt.title(f"Señal ECG extraída (Mala) - {filename}")
    plt.xlabel("Posición horizontal (px)")
    plt.ylabel("Posición vertical media (px)")
    plt.grid()
    
    plt.tight_layout()
    plt.show()  

In [ ]:

# Superposición de varias señales para comparar su forma general
plt.figure(figsize=(12, 6))
for num in range(3):
    signal = señales_malas[num]['signal']
    plt.plot(signal, alpha=0.5)
plt.title("Comparación de señales ECG extraídas (primeras 10 imágenes)")
plt.xlabel("Posición horizontal (px)")
plt.ylabel("Posición vertical media (px)")
plt.grid()
plt.legend([señales_malas[num]['filename'] for num in range(10)], loc='upper right', fontsize='small') 
plt.show()